<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 4 — Exercises: LangChain Basics

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Practise

Five short exercises on the LangChain pieces from today. **No vector database, no embeddings, no
retrieval** — just the four building blocks, and then a real UI on top of them.

| # | What you build |
|---|---|
| **Q1** | your first `ChatOpenAI` call |
| **Q2** | a conversation made of message objects |
| **Q3** | a reusable `ChatPromptTemplate` |
| **Q4** | your first LCEL chain — `prompt \| llm \| parser` |
| **Q5** | a Gradio chat app running on that chain |

**Each cell gives you the steps — you write the code.** Installs, imports and your API key are set up
for you at the top. Run a cell with **Shift + Enter**.

---

## Setup

Run these two cells. Nothing to write.

In [ ]:
# PROVIDED - just run this cell.
!pip install -q langchain langchain-openai gradio

In [ ]:
# PROVIDED - just run this cell.
import os
from getpass import getpass

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import gradio as gr

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
MODEL = "gpt-4o-mini"

print("Setup complete")

---

## Q1 — Your first `ChatOpenAI` call

`ChatOpenAI` is LangChain's wrapper around the model. One object, one method: **`.invoke()`**.

In [ ]:
# Hint: build the model with  ChatOpenAI(model=MODEL, temperature=0)
#       Then call  .invoke("your question")  with a plain string.
#       What comes back is an AIMessage OBJECT, not a string - the text is on  .content
#
# Ask it: "What is LangChain, in two sentences?"
# Print the answer, and then print  type(...)  of whatever .invoke() gave you.


# your code here

That `type()` matters. `.invoke()` hands back an **`AIMessage`**, not a string — which is why
`.content` is there, and why Q4 will need a parser.

---

## Q2 — A conversation, built from messages

Instead of a plain string, `.invoke()` also takes a **list of message objects**:

| Message | Role | What it's for |
|---|---|---|
| `SystemMessage` | system | sets the behaviour |
| `HumanMessage` | user | what the user said |
| `AIMessage` | assistant | what the model said |

In [ ]:
# Hint: build a LIST of message objects instead of a plain string:
#         SystemMessage(content=...)  sets the behaviour
#         HumanMessage(content=...)   asks the question
#       .invoke(that list) returns an AIMessage, same as Q1.
#
# 1. System: "You are a helpful engineering tutor. Keep answers to one sentence."
#    Human:  "What is a prompt template?"
#    Invoke and print the answer.
#
# 2. Now continue the conversation. Append the AIMessage you just got back to your list,
#    append a new HumanMessage("Give me one example."), and invoke the list again.
#    The follow-up only makes sense if the model can see what was said before.


# your code here

**Look at what "memory" turned out to be.** No database, no framework feature — a Python list that
you appended to. Every chatbot you have ever used is doing this, and re-sending the whole list on
every single turn.

---

## Q3 — A reusable prompt

Hardcoding prompts is brittle. A **`ChatPromptTemplate`** separates the prompt's *wording* from its
*data*, using `{variables}` you fill in at call time.

In [ ]:
# Hint: ChatPromptTemplate.from_template("...{variable}...")  builds a reusable prompt.
#       .input_variables  tells you which variables it is expecting.
#       .invoke({...})  fills the template in and returns the MESSAGES - it does NOT call the model.
#
# 1. Write a template with TWO variables: explain {topic} to a {level} student in 2 lines.
# 2. Print  .input_variables
# 3. Fill it in with a topic and a level of your choice, and print the result.
#    Read it - no API call has happened yet, and nothing has been billed.
# 4. Now hand that filled-in prompt to  llm.invoke(...)  and print the answer.


# your code here

Step 3 is the one worth remembering: **you can inspect a prompt before you pay for it.** When an
answer comes out wrong, print the filled-in prompt first — most of the time the bug is visible right
there.

---

## Q4 — Your first LCEL chain

In Q3 you passed the prompt's output to the model by hand. **LCEL** joins them with a pipe:

```
prompt  |  llm  |  StrOutputParser()
```

Every piece speaks `.invoke()`, so `a | b` just means *"run `a`, hand the result to `b`"*.

In [ ]:
# Hint: the pipe order is always  prompt | llm | parser
#       StrOutputParser() takes the AIMessage and hands back a plain str.
#       A chain is invoked exactly like a prompt is:  chain.invoke({...})
#
# 1. Build a chain out of your Q3 prompt, the llm, and StrOutputParser().
#    Invoke it with a topic and a level. Print the result AND its type().
#
# 2. Now build the same chain WITHOUT the parser on the end. Invoke it and print
#    the type() of what you get back.
#    Put the two types side by side - what exactly was the parser doing?


# your code here

`StrOutputParser()` is doing exactly one thing: `AIMessage` → `str`. Small, but it is what lets the
next link in a chain be an ordinary Python function that expects text.

---

## Q5 — Put it behind a chat window

A notebook cell is not a product. Same chain, real UI.

In [ ]:
# Build a Gradio chat app running on an LCEL chain.
#
# 1. Build a small chat chain: a ChatPromptTemplate with a system line that gives the
#    assistant a personality, and a {question} variable for whatever the user types -
#    then pipe it through the llm and a parser.
# 2. Write a function  chat(message, history)  that runs the chain on the user's message
#    and returns the answer.
# 3. Hand that function to gr.ChatInterface, give it a title, and launch it with a
#    public share link.
# 4. Open the link and ask it a few questions.
#
# Then ask a follow-up that depends on your previous question - "and why?" - and watch
# what happens. Look at what your chat() function does with `history`.


# your code here

**`history` arrived and you ignored it.** Gradio hands your function the whole conversation, but the
chain only ever sees `message` — so the model has no idea what was said a moment ago.

You already know the fix, from **Q2**: memory is a growing list of messages. Turning `history` into
`HumanMessage` / `AIMessage` objects and passing the whole list is all it takes.

---

### ✅ What you practised

| Piece | The one line | What comes back |
|---|---|---|
| **`ChatOpenAI`** | `ChatOpenAI(model=..., temperature=0).invoke("...")` | an `AIMessage` → `.content` |
| **Messages** | `[SystemMessage(...), HumanMessage(...)]` | append the reply to continue |
| **`ChatPromptTemplate`** | `.from_template("...{topic}...")` | `.invoke({...})` fills it, no API call |
| **LCEL** | `prompt \| llm \| StrOutputParser()` | a plain `str` |
| **Gradio** | `gr.ChatInterface(chat).launch(share=True)` | a link you can open on your phone |

**Finished early?**

1. Give Q5's `chat()` the conversation: turn `history` into message objects and pass the whole list to the chain. Then re-try the follow-up.
2. Set `temperature=1.5` on the model in Q4 and invoke the same chain three times. Then put it back to `0` and do it again.
3. Add a third variable to the Q3 template — `{language}` — and get the answer in Hindi.